# Atelier 1.C — Tri des demandes clients ★

**Durée : 50 min** · Projet fil rouge, incrément du Jour 1

Fictiva SA reçoit chaque jour des centaines d'emails. Vous construisez le **classifieur** qui attribue à chaque demande une catégorie et un niveau d'urgence, en appliquant les techniques retenues au laboratoire de prompting (1.B).

Livrables :
- un prompt système versionné dans `prompts/tri_demandes.md` ;
- son exactitude mesurée sur le jeu de test.

In [1]:
import outils_llm as llm

MODELE = llm.choisir_modele()  # ou forcez, par ex. "groq:openai/gpt-oss-120b", "nvidia:nvidia/nemotron-3.5-lightning-30b-a3b"
N_MAX = 20                     # les paliers gratuits limitent le débit ; None pour tout le jeu de test
print("Modèle utilisé :", MODELE)

Modèle utilisé : groq:openai/gpt-oss-20b


## 1. Les données

In [2]:
df_ex = llm.charger_demandes("exemples")    # 8 demandes réservées aux exemples du prompt
df_test = llm.charger_demandes("test")      # 40 demandes pour mesurer
print(len(df_ex), "exemples,", len(df_test), "demandes de test")
df_test.groupby(["categorie", "urgence"]).size().unstack(fill_value=0)

8 exemples, 40 demandes de test


urgence,basse,haute,normale
categorie,,,
autre,5,0,1
litige_facturation,0,3,6
question_produit,5,1,3
reclamation,1,3,4
remboursement,0,2,6


In [3]:
print(llm.formater_demande(df_test.iloc[0]))
print("\nAnnotation :", df_test.iloc[0].categorie, "/", df_test.iloc[0].urgence)

<demande>
De : Jonas Wenger (Wenger Énergie)
Objet : Consommation électrique LX-500

Bonjour,

Quelle est la consommation électrique de l'imprimante LX-500 en veille ? Nous préparons notre bilan carbone annuel.

Jonas Wenger
</demande>

Annotation : question_produit / basse


Les règles de tri de Fictiva sont dans `donnees/regles_tri.md`. Lisez-les : sans elles, un modèle ne peut pas deviner ce que Fictiva considère comme « urgent ».

In [4]:
REGLES = llm.lire("regles_tri.md")
print(REGLES)

# Fictiva SA — Règles de tri des demandes clients

Fictiva SA est une entreprise **fictive** de distribution B2B de matériel informatique et de fournitures de bureau (Suisse romande). Le service client reçoit chaque jour plusieurs centaines d'emails. Chaque demande doit recevoir **une catégorie** et **un niveau d'urgence**.

## Catégories

| Catégorie | Définition |
|---|---|
| `reclamation` | Le client signale un problème (produit défectueux ou endommagé, colis abîmé, erreur ou retard de livraison, intervention non effectuée), qu'il demande ou non une correction : remplacement, échange, intervention, livraison du manquant. |
| `remboursement` | Le client demande explicitement un remboursement, un avoir ou un retour de marchandise contre remboursement. |
| `litige_facturation` | Le client conteste une facture ou une relance : montant, prix, remise, TVA, doublon, conditions de paiement, adresse de facturation, blocage de compte lié à une facture. |
| `question_produit` | Demande d'infor

## 2. Le schéma de sortie

Une sortie structurée garantit des valeurs exploitables : `Literal` limite la catégorie et l'urgence aux valeurs autorisées.
Le champ `justification` est placé **en premier** : le modèle écrit sa raison avant de choisir, ce qui améliore souvent le classement et facilite l'analyse des erreurs.

In [5]:
from typing import Literal
from pydantic import BaseModel, Field

Categorie = Literal["reclamation", "remboursement", "litige_facturation", "question_produit", "autre"]
Urgence = Literal["haute", "normale", "basse"]

class Tri(BaseModel):
    justification: str = Field(description="Une phrase citant l'élément décisif de la demande")
    categorie: Categorie
    urgence: Urgence

## 3. Version 1 du prompt

Le prompt applique les techniques de l'atelier 1.B : rôle, contexte (les règles, entre balises), exemples tirés de `df_ex`, méthode de décision.
Le format de sortie est déjà imposé par le schéma : inutile de le décrire en détail.

✏️ Comparez avec vos conclusions du laboratoire 1.B et modifiez le prompt si vous le souhaitez.

In [6]:
def construire_exemples(df):
    """Transforme les demandes d'exemple en texte few-shot."""
    blocs = []
    for _, ligne in df.iterrows():
        blocs.append(
            f"{llm.formater_demande(ligne)}\n"
            f"<reponse>categorie={ligne['categorie']} ; urgence={ligne['urgence']}</reponse>"
        )
    return "\n\n".join(blocs)

PROMPT_V1 = f"""Tu es chargé du tri des emails entrants du service client de Fictiva SA, distributeur B2B de matériel informatique et de fournitures de bureau.

Pour chaque demande, attribue une catégorie et un niveau d'urgence en appliquant strictement les règles ci-dessous.

<regles>
{REGLES}
</regles>

<exemples>
{construire_exemples(df_ex)}
</exemples>

Méthode :
1. Repère l'intention principale du client ; s'il y en a plusieurs, applique l'ordre de priorité des règles.
2. Vérifie un par un les critères d'urgence haute, puis ceux d'urgence basse ; sinon, l'urgence est normale.
3. Dans la justification, cite en une phrase l'élément de la demande qui a décidé du classement."""
print(PROMPT_V1[:1500], "…")

Tu es chargé du tri des emails entrants du service client de Fictiva SA, distributeur B2B de matériel informatique et de fournitures de bureau.

Pour chaque demande, attribue une catégorie et un niveau d'urgence en appliquant strictement les règles ci-dessous.

<regles>
# Fictiva SA — Règles de tri des demandes clients

Fictiva SA est une entreprise **fictive** de distribution B2B de matériel informatique et de fournitures de bureau (Suisse romande). Le service client reçoit chaque jour plusieurs centaines d'emails. Chaque demande doit recevoir **une catégorie** et **un niveau d'urgence**.

## Catégories

| Catégorie | Définition |
|---|---|
| `reclamation` | Le client signale un problème (produit défectueux ou endommagé, colis abîmé, erreur ou retard de livraison, intervention non effectuée), qu'il demande ou non une correction : remplacement, échange, intervention, livraison du manquant. |
| `remboursement` | Le client demande explicitement un remboursement, un avoir ou un retour de 

## 4. Lancer le tri sur le jeu de test

In [7]:
def trier(ligne, prompt):
    return llm.demander(llm.formater_demande(ligne), systeme=prompt, modele=MODELE, schema=Tri)

echantillon = df_test if N_MAX is None else df_test.head(N_MAX)
lignes = [ligne for _, ligne in echantillon.iterrows()]
reponses_v1 = llm.en_lot(lambda ligne: trier(ligne, PROMPT_V1), lignes)

20 appels terminés, 0 erreur(s).


## 5. Mesurer

In [8]:
def tableau(echantillon, reponses):
    """Annotations, prédictions et justifications côte à côte."""
    res = echantillon[["id", "objet", "categorie", "urgence"]].copy()
    ok = [isinstance(r, llm.Reponse) and r.objet is not None for r in reponses]
    res["categorie_predite"] = [r.objet.categorie if o else None for r, o in zip(reponses, ok)]
    res["urgence_predite"] = [r.objet.urgence if o else None for r, o in zip(reponses, ok)]
    res["justification"] = [r.objet.justification if o else str(r) for r, o in zip(reponses, ok)]
    return res

def mesurer(res, reponses, version):
    bonne_categorie = res.categorie == res.categorie_predite
    bonne_urgence = res.urgence == res.urgence_predite
    return {
        "version": version,
        "exactitude_categorie": round(bonne_categorie.mean(), 3),
        "exactitude_urgence": round(bonne_urgence.mean(), 3),
        "exactitude_globale": round((bonne_categorie & bonne_urgence).mean(), 3),
        **llm.bilan(reponses),
    }

res_v1 = tableau(echantillon, reponses_v1)
mesures_v1 = mesurer(res_v1, reponses_v1, "v1")
mesures_v1

{'version': 'v1',
 'exactitude_categorie': np.float64(1.0),
 'exactitude_urgence': np.float64(0.95),
 'exactitude_globale': np.float64(0.95),
 'appels': 20,
 'latence_moyenne_s': 20.73,
 'tokens_entree_moyens': 1962,
 'tokens_sortie_moyens': 222,
 'cout_pour_1000_usd': 0.214}

In [9]:
llm.matrice_confusion(res_v1.categorie, res_v1.categorie_predite)

prédiction,autre,litige_facturation,question_produit,reclamation,remboursement
vérité,,,,,
autre,4,0,0,0,0
litige_facturation,0,3,0,0,0
question_produit,0,0,6,0,0
reclamation,0,0,0,2,0
remboursement,0,0,0,0,5


In [10]:
llm.matrice_confusion(res_v1.urgence, res_v1.urgence_predite)

prédiction,basse,haute,normale
vérité,,,
basse,8,0,0
haute,0,4,1
normale,0,0,7


## 6. Analyser les erreurs

Lisez chaque erreur et sa justification. Pour chacune, demandez-vous : la règle était-elle ambiguë ? Le modèle l'a-t-il mal appliquée ? L'annotation elle-même est-elle discutable ?

In [11]:
def afficher_erreurs(res):
    erreurs = res[(res.categorie != res.categorie_predite) | (res.urgence != res.urgence_predite)]
    print(len(erreurs), "erreur(s)\n")
    for _, e in erreurs.iterrows():
        print(f"{e.id} | {e.objet}")
        print(f"   attendu : {e.categorie} / {e.urgence}    prédit : {e.categorie_predite} / {e.urgence_predite}")
        print(f"   justification : {e.justification}\n")

afficher_erreurs(res_v1)

1 erreur(s)

DEM-004 | Appel d'offres – renouvellement du parc
   attendu : question_produit / haute    prédit : question_produit / normale
   justification : demande d'information tarifaire et fiches techniques avant jeudi midi



## 7. Version 2

Les précisions ci-dessous répondent aux confusions les plus fréquentes : seuil de montant, activité bloquée, question assortie d'un délai, signalement sans demande, demande d'argent. Elles sont **générales** : elles valent pour toute demande future.

✏️ Adaptez-les aux erreurs que **vous** avez observées.

> Attention au surapprentissage : ne recopiez pas les demandes du jeu de test dans les exemples. En production, on garderait un troisième jeu, jamais consulté, pour la mesure finale.

In [12]:
PROMPT_V2 = PROMPT_V1 + """

Points d'attention :
- Le montant en jeu peut être un budget, une commande, un remboursement ou une facture : dès qu'il dépasse 5 000 CHF, l'urgence est haute, quelle que soit la catégorie.
- Une contestation de facture qui empêche le client de travailler ou de commander relève de l'urgence haute (activité bloquée).
- Une question assortie d'un délai ou d'une échéance proche appelle une réponse rapide : urgence normale, pas basse.
- Un problème signalé sans demande d'action reste une reclamation, d'urgence basse.
- Dès que le client demande de l'argent (remboursement, avoir, trop-perçu à rendre), la catégorie est remboursement, même si une facture est citée."""

In [13]:
reponses_v2 = llm.en_lot(lambda ligne: trier(ligne, PROMPT_V2), lignes)
res_v2 = tableau(echantillon, reponses_v2)
mesures_v2 = mesurer(res_v2, reponses_v2, "v2")

import pandas as pd
pd.DataFrame([mesures_v1, mesures_v2]).set_index("version")

20 appels terminés, 0 erreur(s).


,exactitude_categorie,exactitude_urgence,exactitude_globale,appels,latence_moyenne_s,tokens_entree_moyens,tokens_sortie_moyens,cout_pour_1000_usd
version,,,,,,,,
v1,1.0,0.95,0.95,20,20.73,1962,222,0.214
v2,1.0,0.85,0.85,20,16.68,2112,302,0.249


In [14]:
afficher_erreurs(res_v2)

3 erreur(s)

DEM-003 | Étude de cas – digitalisation des PME
   attendu : autre / basse    prédit : autre / normale
   justification : demande d'entretien d'étude, non liée à un produit ou service

DEM-004 | Appel d'offres – renouvellement du parc
   attendu : question_produit / haute    prédit : question_produit / normale
   justification : demande d'information tarifaire et fiches techniques avant jeudi midi

DEM-020 | Candidature spontanée – logisticienne
   attendu : autre / basse    prédit : autre / normale
   justification : La demande est une candidature spontanée, ce qui ne correspond à aucune catégorie spécifique mais relève du champ "autre".



## 8. Versionner le prompt retenu

Un prompt est du code : on le versionne avec ses résultats. Le fichier enregistré sert à l'atelier 1.E et sera repris au Jour 3 par l'agent du projet fil rouge.

In [15]:
from datetime import date

meilleure, mesures = (
    (PROMPT_V2, mesures_v2)
    if mesures_v2["exactitude_globale"] >= mesures_v1["exactitude_globale"]
    else (PROMPT_V1, mesures_v1)
)
chemin = llm.sauver_prompt(
    "tri_demandes",
    meilleure,
    version=mesures["version"],
    date=date.today().isoformat(),
    modele=MODELE,
    exactitude_globale=mesures["exactitude_globale"],
    demandes_testees=len(echantillon),
)
print("Prompt enregistré :", chemin)

Prompt enregistré : /Users/bassembenhamed/Desktop/Agents IA & RPA/Module 01 - LLM/ateliers/prompts/tri_demandes.md


## À retenir

- Les **règles métier** dans le contexte font la plus grande différence : le modèle ne peut pas deviner les critères d'urgence de Fictiva.
- Un schéma de sortie supprime les erreurs de format ; la justification rend les erreurs lisibles.
- On améliore un prompt par mesure, analyse des erreurs et correction **générale**, sans surapprendre le jeu de test.